# 🧠 BrainScan — Détection de tumeurs cérébrales par CNN (version Kaggle)

**Avant de lancer :**
1. **Add Input** → chercher le dataset **"Brain Tumor MRI Dataset"** (Masoud Nickparvar) → *Add*.
2. **Settings → Accelerator → GPU T4 x2** (ou P100) pour un entraînement beaucoup plus rapide.
3. **Run All**.

Le notebook est autonome : la classe `Preprocessor` est écrite dans la cellule 2 (pas besoin d'uploader `preprocessing.py`).

In [ ]:
%%writefile preprocessing.py
import numpy as np
import cv2 as cv
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

class Preprocessor:
    def __init__(self, image_size=(224, 224), path=None, classetr=None):

        self.image_size = image_size

        self.path = Path(path) if path is not None else None

        self.classetr = classetr if classetr is not None else []

        self.images = np.array([])
        self.labels = np.array([])

        self.valid_extensions = {
            ".jpg", ".jpeg", ".png", ".bmp", ".webp"
        }

        self._X_train = None
        self._X_test = None
        self._y_train = None
        self._y_test = None
        self._X_val = None
        self._y_val = None

    def load_images(self):
        images = []
        labels = []

        if self.path is None:
            raise ValueError("Le chemin du dataset n'est pas défini.")

        if not self.path.exists():
            raise FileNotFoundError(
                f"Le dossier du dataset n'existe pas : {self.path}"
            )

        for class_name in self.classetr:
            class_path = self.path / class_name

            if not class_path.is_dir():
                print(f"Warning: Dossier introuvable : {class_path}")
                continue

            for image_file in class_path.iterdir():

                if image_file.suffix.lower() not in self.valid_extensions:
                    print(f"Extension invalide : {image_file}")
                    continue

                if not image_file.is_file():
                    continue

                image = cv.imread(str(image_file))

 
                if image is None or image.size == 0:
                    print(f"Image illisible ou corrompue : {image_file}")
                    continue


                image = cv.resize(image, self.image_size)

                image = cv.cvtColor(image, cv.COLOR_BGR2RGB)


                images.append(image)
                labels.append(class_name)


        if images:
            self.images = np.array(images, dtype=np.uint8)
        else:
            self.images = np.empty(
                (0, self.image_size[1], self.image_size[0], 3),
                dtype=np.uint8
            )

        self.labels = np.array(labels)

        return self.images, self.labels

    def encode_labels(self):
        if self.labels.size == 0:
            raise ValueError("Aucun label à encoder. Chargez d'abord les images.")

        label_encoder = LabelEncoder()
        encoded_labels = label_encoder.fit_transform(self.labels)
        self.labels = encoded_labels

        return self.labels

    def normalize_images(self):
        if self.images.size == 0:
            raise ValueError("Aucune image à normaliser. Chargez d'abord les images.")

        self.images = self.images.astype(np.float32) / 255.0

    def split_data(self, test_size=0.2, random_state=42):
        if self.images.size == 0 or self.labels.size == 0:
            raise ValueError("Aucune donnée à diviser. Chargez d'abord les images et les labels.")

        X_train, X_test, y_train, y_test = train_test_split(
            self.images,
            self.labels,
            test_size=test_size,
            random_state=random_state,
            stratify=self.labels
        )

        self._X_train = X_train
        self._X_test = X_test
        self._y_train = y_train
        self._y_test = y_test

        return X_train, X_test, y_train, y_test

    def get_train_data(self):
        if self._X_train is None or self._y_train is None:
            raise ValueError("Les données d'entraînement n'ont pas été générées. Appelez d'abord split_data().")
        return self._X_train, self._y_train
    def get_test_data(self):
        if self._X_test is None or self._y_test is None:
            raise ValueError("Les données de test n'ont pas été générées. Appelez d'abord split_data().")
        return self._X_test, self._y_test

    def get_validation_data(self):
        if self._X_val is None or self._y_val is None:
            raise ValueError("Les données de validation n'ont pas été générées. Appelez d'abord split_data_validation().")
        return self._X_val, self._y_val

    def split_data_validation(self):
        if self._X_train is None or self._y_train is None:
            raise ValueError("Les données d'entraînement n'ont pas été générées. Appelez d'abord split_data().")

        X_train, X_val, y_train, y_val = train_test_split(
            self._X_train,
            self._y_train,
            test_size=0.2,
            random_state=42,
            stratify=self._y_train
        )

        self._X_train = X_train
        self._X_val = X_val
        self._y_train = y_train
        self._y_val = y_val

        

        


    
if __name__ == "__main__":

    preprocessor = Preprocessor(
        image_size=(224, 224),
        path="./Data",
        classetr=[
            "glioma",
            "meningioma",
            "pituitary",
            "notumor"
        ]
    )

    images, labels = preprocessor.load_images()

    preprocessor.encode_labels()
    preprocessor.normalize_images()
    preprocessor.split_data(test_size=0.2, random_state=42)
    X_train, y_train = preprocessor.get_train_data()
    X_test, y_test = preprocessor.get_test_data()

    print("X_train :", X_train.shape)
    print("X_test  :", X_test.shape)

    print("y_train :", y_train.shape)
    print("y_test  :", y_test.shape)

In [ ]:
import os, sys, random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

sys.path.insert(0, os.getcwd())
from preprocessing import Preprocessor

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices("GPU"))

## 1. Localiser le dataset et fusionner Training + Testing
Le dataset Kaggle est déjà séparé en `Training/` et `Testing/`. Comme ton `Preprocessor` fait son propre split, on regroupe tout dans un dossier `Data/<classe>/` (liens symboliques, pas de copie lourde).

In [ ]:
CLASSES = ["glioma", "meningioma", "pituitary", "notumor"]

# 1) trouver les dossiers qui contiennent les 4 classes
split_dirs = sorted({p.parent for p in Path("/kaggle/input").rglob("glioma") if p.is_dir()})
print("Dossiers trouvés :", split_dirs)
assert split_dirs, "Dataset introuvable : ajoute 'Brain Tumor MRI Dataset' via Add Input."

# 2) fusionner dans /kaggle/working/Data/<classe>/
DATA_DIR = Path("/kaggle/working/Data")
for cls in CLASSES:
    (DATA_DIR / cls).mkdir(parents=True, exist_ok=True)
    for sd in split_dirs:
        for f in (sd / cls).iterdir():
            link = DATA_DIR / cls / f"{sd.name}_{f.name}"
            if not link.exists():
                link.symlink_to(f)

for cls in CLASSES:
    print(f"{cls:12s}", len(list((DATA_DIR / cls).iterdir())), "images")

## 2. Chargement des images (avec ta classe `Preprocessor`)

In [ ]:
IMG_SIZE = (224, 224)

preprocessor = Preprocessor(image_size=IMG_SIZE, path=str(DATA_DIR), classetr=CLASSES)
images, labels = preprocessor.load_images()
print("images :", images.shape, images.dtype)
print("labels :", labels.shape)

## 3. Exploration : distribution des classes et exemples

In [ ]:
label_counts = Counter(labels)
cats, counts = list(label_counts.keys()), list(label_counts.values())

plt.figure(figsize=(9, 4))
plt.bar(cats, counts, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
for i, c in enumerate(counts):
    plt.text(i, c, str(c), ha="center", va="bottom")
plt.title("Nombre d'images par catégorie"); plt.ylabel("Nombre d'images")
plt.tight_layout(); plt.show()

In [ ]:
cats_sorted = np.unique(labels)
fig, axes = plt.subplots(len(cats_sorted), 4, figsize=(12, 3 * len(cats_sorted)))
for i, cat in enumerate(cats_sorted):
    idx = np.where(labels == cat)[0][:4]
    for j, k in enumerate(idx):
        axes[i, j].imshow(images[k]); axes[i, j].set_title(cat); axes[i, j].axis("off")
plt.tight_layout(); plt.show()

## 4. Encodage, normalisation et split Train / Validation / Test

In [ ]:
preprocessor.encode_labels()
preprocessor.normalize_images()
preprocessor.split_data(test_size=0.2, random_state=SEED)
preprocessor.split_data_validation()

X_train, y_train = preprocessor.get_train_data()
X_val,   y_val   = preprocessor.get_validation_data()
X_test,  y_test  = preprocessor.get_test_data()

# LabelEncoder trie par ordre alphabétique
class_names = sorted(CLASSES)
num_classes = len(class_names)
print("Classes :", dict(enumerate(class_names)))
print("Train :", X_train.shape, "| Val :", X_val.shape, "| Test :", X_test.shape)

In [ ]:
# Équilibrage : poids par classe (utile si les classes sont déséquilibrées)
weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(weights))
print("class_weights :", {class_names[k]: round(v, 3) for k, v in class_weights.items()})

## 5. Modèle CNN
Même base que ton modèle (Conv2D → MaxPooling → Dense), avec `BatchNormalization` et `Dropout` pour réduire l'overfitting.

In [ ]:
def build_model(input_shape=(224, 224, 3), num_classes=4):
    return keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])

model = build_model(IMG_SIZE + (3,), num_classes)
model.summary()

## 6. Entraînement
`ModelCheckpoint` garde le meilleur modèle, `EarlyStopping` arrête si la `val_loss` ne s'améliore plus, `ReduceLROnPlateau` baisse le learning rate.

In [ ]:
LEARNING_RATE = 1e-3
EPOCHS        = 30
BATCH_SIZE    = 32

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",   # labels entiers (LabelEncoder)
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.ModelCheckpoint("/kaggle/working/best_model.keras",
                                    monitor="val_accuracy", save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
)

## 7. Courbes d'apprentissage

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(history.history["accuracy"], label="train")
ax[0].plot(history.history["val_accuracy"], label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()

ax[1].plot(history.history["loss"], label="train")
ax[1].plot(history.history["val_loss"], label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Évaluation sur le jeu de test

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss : {test_loss:.4f} | Test accuracy : {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test, batch_size=64), axis=1)
print()
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45)
plt.yticks(range(num_classes), class_names)
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("Prédit"); plt.ylabel("Réel"); plt.title("Matrice de confusion")
plt.tight_layout(); plt.show()

## 9. Sauvegarde du modèle et fonction `predict_image`
Les fichiers sauvegardés dans `/kaggle/working/` sont téléchargeables depuis l'onglet **Output** (à droite) après *Save Version*.

In [ ]:
model.save("/kaggle/working/brainscan_model.keras")
model.save("/kaggle/working/brainscan_model.h5")   # format demandé dans le brief
print(os.listdir("/kaggle/working"))

In [ ]:
def predict_image(path, model=model, class_names=class_names, image_size=IMG_SIZE, show=True):
    img = cv.imread(str(path))
    if img is None:
        raise ValueError(f"Image illisible : {path}")
    img = cv.cvtColor(cv.resize(img, image_size), cv.COLOR_BGR2RGB)
    x = np.expand_dims(img.astype("float32") / 255.0, axis=0)

    probs = model.predict(x, verbose=0)[0]
    k = int(np.argmax(probs))

    if show:
        plt.imshow(img); plt.axis("off")
        plt.title(f"{class_names[k]} ({probs[k]*100:.1f}%)"); plt.show()
    return class_names[k], float(probs[k])

# Exemple sur une image du jeu de test
sample = next((DATA_DIR / "glioma").iterdir())
print(predict_image(sample))